In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import os

In [2]:
adata = ad.read_h5ad("../03_output/01_data_deposition_object_preparation/myeloid_cells.h5ad")
adata.obs['assay_ontology_term_id'] = adata.obs['assay']
adata.write_h5ad("../03_output/01_data_deposition_object_preparation/myeloid_cells.h5ad")

In [3]:
adata = ad.read_h5ad("../03_output/01_data_deposition_object_preparation/tnk_cells.h5ad")
adata.obs['assay_ontology_term_id'] = adata.obs['assay']
adata.obs['cell_type_ontology_term_id'] = (
    adata.obs['cell_type_ontology_term_id']
    .astype(str)
    .replace({"CL:0020002": "CL:0000940"})
)
adata.write_h5ad("../03_output/01_data_deposition_object_preparation/tnk_cells.h5ad")

In [15]:
adata.obs['cell_type_ontology_term_id'] = (
    adata.obs['cell_type_ontology_term_id']
    .astype(str)
    .replace({"CL:0020002": "CL:0000940"})
)

In [18]:
adata.write_h5ad("../03_output/01_data_deposition_object_preparation/scRNAseq_atlas.h5ad")

In [4]:
datasets = [
    {
        "name": "snRNAseq_vics",
        "h5ad": "../03_output/01_data_deposition_object_preparation/snRNAseq_vics.h5ad",
        "magic": "../../02_UBC_snRNA_seq/03_output/04_UBC_snRNAseq_vic_annotation/vic_magic_output.csv",
        "title": "snRNA-seq VICs — Cellular and Molecular Pathogenesis of Aortic Stenosis",
        "suspension": "nucleus",
        "annotation_levels": ["annotations_level1", "annotations_level2"],
        "default_embedding": "X_umap",
    },
    {
        "name": "snRNAseq_atlas",
        "h5ad": "../03_output/01_data_deposition_object_preparation/snRNAseq_atlas.h5ad",
        "magic": "../03_output/01_data_deposition_object_preparation/snRNAseq_atlas_magic_output.csv",
        "title": "snRNA-seq Atlas — Cellular and Molecular Pathogenesis of Aortic Stenosis",
        "suspension": "nucleus",
        "annotation_levels": ["annotations_level1", "annotations_level2"],
        "default_embedding": "X_umap",
    },
    {
        "name": "scRNAseq_atlas",
        "h5ad": "../03_output/01_data_deposition_object_preparation/scRNAseq_atlas.h5ad",
        "magic": "../03_output/01_data_deposition_object_preparation/scRNAseq_atlas_magic_output.csv",
        "title": "scRNA-seq Atlas — Cellular and Molecular Pathogenesis of Aortic Stenosis",
        "suspension": "cell",
        "annotation_levels": ["annotations_level1", "annotations_level2", "annotations_level3"],
        "default_embedding": "X_paga",
    },
    {
        "name": "endothelial",
        "h5ad": "../03_output/01_data_deposition_object_preparation/endothelial.h5ad",
        "magic": "../03_output/01_data_deposition_object_preparation/magic_output/endothelial_magic_output.csv",
        "title": "scRNA-seq Endothelial Cells — Cellular and Molecular Pathogenesis of Aortic Stenosis",
        "suspension": "cell",
        "annotation_levels": ["annotations_level1", "annotations_level2", "annotations_level3"],
        "default_embedding": "X_umap",
    },
    {
        "name": "vics",
        "h5ad": "../03_output/01_data_deposition_object_preparation/vics.h5ad",
        "magic": "../03_output/01_data_deposition_object_preparation/magic_output/vics_magic_output.csv",
        "title": "scRNA-seq VICs — Cellular and Molecular Pathogenesis of Aortic Stenosis",
        "suspension": "cell",
        "annotation_levels": ["annotations_level1", "annotations_level2", "annotations_level3"],
        "default_embedding": "X_umap",
    },
    {
        "name": "myeloid_cells",
        "h5ad": "../03_output/01_data_deposition_object_preparation/myeloid_cells.h5ad",
        "magic": "../03_output/01_data_deposition_object_preparation/magic_output/myeloid_cells_magic_output.csv",
        "title": "scRNA-seq Myeloid Cells — Cellular and Molecular Pathogenesis of Aortic Stenosis",
        "suspension": "cell",
        "annotation_levels": ["annotations_level1", "annotations_level2", "annotations_level3"],
        "default_embedding": "X_umap",
    },
    {
        "name": "tnk_cells",
        "h5ad": "../03_output/01_data_deposition_object_preparation/tnk_cells.h5ad",
        "magic": "../03_output/01_data_deposition_object_preparation/magic_output/tnk_cells_magic_output.csv",
        "title": "scRNA-seq T-NK Cells — Cellular and Molecular Pathogenesis of Aortic Stenosis",
        "suspension": "cell",
        "annotation_levels": ["annotations_level1", "annotations_level2", "annotations_level3"],
        "default_embedding": "X_umap",
    },
]

In [ ]:
# =============================================
# PROCESSING FUNCTION
# =============================================
def prepare_cellxgene(config):
    print(f"\n{'='*60}")
    print(f"Processing: {config['name']}")
    print(f"{'='*60}")

    adata = ad.read_h5ad(config["h5ad"])
    print(f"Loaded: {adata.shape}")

    # --- 1. KEEP ORIGINAL GENE IDS ---
    print(f"Keeping original gene IDs in adata.var.index")
    print(f"Number of duplicated gene IDs: {adata.var.index.duplicated().sum()}")
    
    # --- 2. CLEAN VAR ---
    adata.var = adata.var[[]]
    adata.var['feature_is_filtered'] = False

    # --- 3. X / RAW ---
    raw_adata = ad.AnnData(
        X=sp.csr_matrix(adata.layers['counts'].astype(np.float32)),
        var=adata.var[[]].copy(),
        obs=adata.obs.copy()
    )

    adata.X = sp.csr_matrix(adata.layers['data'].astype(np.float32))
    adata.raw = raw_adata

    del adata.layers['counts']
    del adata.layers['data']

    # --- 4. MAGIC LAYER ---
    if config["magic"] is not None:
        magic = pd.read_csv(config["magic"], index_col=0)
        magic.index = magic.index.str.strip()
        magic.columns = magic.columns.str.strip()

        # Match cell order
        magic = magic.loc[adata.obs.index]

        # Do not rename MAGIC columns.
        # Match directly to the existing gene IDs in adata.var.index.
        magic_full = pd.DataFrame(
            0.0,
            index=adata.obs.index,
            columns=adata.var.index
        )

        var_genes = set(adata.var.index)
        magic_cols_in_var = [g for g in magic.columns if g in var_genes]

        magic_full[magic_cols_in_var] = magic[magic_cols_in_var].values

        print(f"MAGIC genes matched: {len(magic_cols_in_var)} / {magic.shape[1]}")

        adata.layers['MAGIC_imputed'] = sp.csr_matrix(
            magic_full.values.astype(np.float32)
        )
    else:
        print("No MAGIC layer for this dataset")

    # --- 5. OBS ---
    obs_keep = [
        'donor_id', 'clinical_classification',
    ] + config["annotation_levels"] + [
        'development_stage_ontology_term_id',
        'self_reported_ethnicity_ontology_term_id',
        'sex_ontology_term_id',
        'disease_ontology_term_id',
        'tissue_type',
        'tissue_ontology_term_id',
        'cell_type_ontology_term_id',
        'assay_ontology_term_id',
        'suspension_type',
    ]

    # Only keep columns that exist in this object
    obs_keep = [c for c in obs_keep if c in adata.obs.columns]
    adata.obs = adata.obs[obs_keep].copy()
    adata.obs['is_primary_data'] = True

    # Fix suspension_type
    adata.obs['suspension_type'] = (
        adata.obs['suspension_type']
        .astype(str)
        .replace({"nuclei": "nucleus", "cells": "cell"})
        .astype('category')
    )

    for col in obs_keep:
        adata.obs[col] = adata.obs[col].astype('category')

    # --- 6. UNS ---
    adata.uns = {}
    adata.uns['title'] = config["title"]
    adata.uns['organism_ontology_term_id'] = "NCBITaxon:9606"
    adata.uns['default_embedding'] = config["default_embedding"]
    adata.uns['batch_condition'] = ["donor_id"]

    # --- 7. OBSM ---
    rename_map = {
        'umap': 'X_umap',
        'pca': 'X_pca',
        'harmony': 'X_harmony',
        'fa': 'X_fa',
        'paga': 'X_paga'
    }

    for old, new in rename_map.items():
        if old in adata.obsm:
            adata.obsm[new] = adata.obsm.pop(old)

    # --- 8. OBSP ---
    for key in list(adata.obsp.keys()):
        del adata.obsp[key]

    # --- 9. SAVE ---
    out_path = f"../03_output/02_cellxgene_output/{config['name']}_cellxgene.h5ad"
    adata.write_h5ad(out_path, compression="gzip")

    print(f"Saved: {out_path} ({adata.shape})")

    return adata


# =============================================
# RUN ALL
# =============================================
os.makedirs("../03_output/02_cellxgene_output", exist_ok=True)

results = {}
for ds in datasets:
    results[ds["name"]] = prepare_cellxgene(ds)


Processing: snRNAseq_vics
Loaded: (53870, 38606)
Keeping original gene IDs in adata.var.index
Number of duplicated gene IDs: 0
MAGIC genes matched: 2000 / 2000
Saved: ../03_output/02_cellxgene_output/snRNAseq_vics_cellxgene.h5ad ((53870, 38606))

Processing: snRNAseq_atlas
Loaded: (62829, 38606)
Keeping original gene IDs in adata.var.index
Number of duplicated gene IDs: 0
MAGIC genes matched: 2000 / 2000
Saved: ../03_output/02_cellxgene_output/snRNAseq_atlas_cellxgene.h5ad ((62829, 38606))

Processing: scRNAseq_atlas
Loaded: (84611, 38606)
Keeping original gene IDs in adata.var.index
Number of duplicated gene IDs: 0
MAGIC genes matched: 2000 / 2000
Saved: ../03_output/02_cellxgene_output/scRNAseq_atlas_cellxgene.h5ad ((84611, 38606))

Processing: endothelial
Loaded: (3585, 38606)
Keeping original gene IDs in adata.var.index
Number of duplicated gene IDs: 0
MAGIC genes matched: 2000 / 2000
Saved: ../03_output/02_cellxgene_output/endothelial_cellxgene.h5ad ((3585, 38606))

Processing: v